In [5]:
import pandas as pd
geo_info = pd.read_parquet("/Users/musa.official/Documents/UoB-GeneTraceAI-25-26/data/parquet/data_clean/geo_info_clean.parquet")

In [7]:
geo_info.tail(3)

,geo_accession,cel_file_names,title,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,...,contact_institute,gse_id,gse_filename,cell_line,disease,origin,cellosaurus_id,cellline,matching_type,cell_line_trimmed
3264,gsm960296,gsm960296,none,none,none,none,none,NaN,none,none,...,none,none,none,none,none,none,cvcl_0038,hacat,cello geo gsm,none
3265,gsm960297,gsm960297,none,none,none,none,none,NaN,none,none,...,none,none,none,none,none,none,cvcl_7082,none,none,none
3266,gsm960298,gsm960298,none,none,none,none,none,NaN,none,none,...,none,none,none,none,none,none,cvcl_7082,none,none,none


In [9]:
hpa_desc = pd.read_parquet("/Users/musa.official/Documents/UoB-GeneTraceAI-25-26/data/parquet/data_clean/hpa_desc_clean.parquet")
hpa_desc.head(3)

,cell line,disease,disease subtype,cellosaurus id,patient,primary/metastasis,sample collection site
0,143b,bone cancer,osteosarcoma,cvcl_2270,13,primary,bone
1,22rv1,prostate cancer,adenocarcinoma,cvcl_1045,male,primary,prostate
2,23132/87,gastric cancer,adenocarcinoma,cvcl_1046,"male, 72",primary,stomach


In [10]:
geo_info.columns

Index(['geo_accession', 'cel_file_names', 'title', 'status', 'submission_date',
       'last_update_date', 'type', 'channel_count', 'source_name_ch1',
       'organism_ch1', 'characteristics_ch1', 'platform_id', 'contact_country',
       'contact_institute', 'gse_id', 'gse_filename', 'cell_line', 'disease',
       'origin', 'cellosaurus_id', 'cellline', 'matching_type',
       'cell_line_trimmed'],
      dtype='object')

In [11]:
geo_expr = pd.read_parquet("/Users/musa.official/Documents/UoB-GeneTraceAI-25-26/data/parquet/data_clean/geo_expr_clean.parquet")
geo_expr.head(3)

,gene,gsm101610,gsm101615,gsm101616,gsm101667,gsm101668,gsm101671,gsm101672,gsm101673,gsm101674,...,gsm960289,gsm960290,gsm960291,gsm960292,gsm960293,gsm960294,gsm960295,gsm960296,gsm960297,gsm960298
0,ensg00000000003,33.615700,553.249756,540.452209,599.431152,625.242737,400.554657,412.995605,427.403900,461.123535,...,197.712616,387.427826,116.657890,104.889442,158.400604,736.519043,631.033142,791.054504,181.986893,353.206940
1,ensg00000000005,40.925682,31.327406,33.934967,34.213123,32.466286,36.243233,37.952511,34.644428,34.225410,...,4.392303,4.230473,4.651096,5.175624,5.540960,4.211264,4.159354,4.480873,4.630062,4.296047
2,ensg00000000419,2182.281250,3419.430420,3514.540039,2295.817383,2378.469727,2297.564697,2276.069092,2368.761475,2689.160156,...,936.201782,1162.099731,1249.029053,1354.346069,1159.747559,1561.643799,1547.089233,1551.400146,939.964661,1054.350952


In [12]:
geo_expr.columns

Index(['gene', 'gsm101610', 'gsm101615', 'gsm101616', 'gsm101667', 'gsm101668',
       'gsm101671', 'gsm101672', 'gsm101673', 'gsm101674',
       ...
       'gsm960289', 'gsm960290', 'gsm960291', 'gsm960292', 'gsm960293',
       'gsm960294', 'gsm960295', 'gsm960296', 'gsm960297', 'gsm960298'],
      dtype='object', length=3268)

In [13]:
import re

# 1. THE decisive check: how many GEO series feed this table?
print("distinct GSE series:", geo_info['gse_id'].nunique())
print(geo_info['gse_id'].value_counts().head(10))

# 2. Confirm single platform (should be all GPL570)
print("\nplatforms:\n", geo_info['platform_id'].value_counts())

# 3. One array per line, or many?  Group on the harmonised identity key.
print("\nrows:", len(geo_info), " distinct lines:", geo_info['cellosaurus_id'].nunique())
print("samples-per-line distribution:")
print(geo_info.groupby('cellosaurus_id').size().value_counts().sort_index())

# 4. Does any free text encode perturbation?
tok = r'treat|drug|dose|µm|\bum\b|vehicle|\bcontrol\b|\d+\s*h(r|our)?\b|stimul|sirna|shrna|knockdown|overexpress|transfect'
for col in ['characteristics_ch1', 'title', 'source_name_ch1']:
    n = geo_info[col].astype(str).str.contains(tok, case=False, regex=True, na=False).sum()
    print(f"{col}: {n} rows with perturbation-like tokens")

# 5. Eyeball what the fields actually say
geo_info[['gse_id','title','source_name_ch1','characteristics_ch1','cell_line']].head(10)

distinct GSE series: 19
gse_id
none        938
gse57083    627
gse50811    241
gse34211    230
gse10843    206
gse50831    189
gse50830    168
gse10890    112
gse23806     80
gse30240     75
Name: count, dtype: int64

platforms:
 platform_id
gpl570    2329
none       938
Name: count, dtype: int64

rows: 3267  distinct lines: 798
samples-per-line distribution:
1      378
2      142
3       74
4       49
5       26
6       20
7       15
8        9
9       25
10       7
11       8
12      10
13       2
14       5
15       7
16       5
17       3
18       1
20       2
21       1
23       1
24       1
25       1
31       1
41       1
49       1
67       1
108      1
478      1
Name: count, dtype: int64
characteristics_ch1: 336 rows with perturbation-like tokens
title: 136 rows with perturbation-like tokens
source_name_ch1: 690 rows with perturbation-like tokens


/var/folders/w4/fbz2zhr165g6wr5k09mtgl9r0000gn/T/ipykernel_66111/4184146311.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  n = geo_info[col].astype(str).str.contains(tok, case=False, regex=True, na=False).sum()
/var/folders/w4/fbz2zhr165g6wr5k09mtgl9r0000gn/T/ipykernel_66111/4184146311.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  n = geo_info[col].astype(str).str.contains(tok, case=False, regex=True, na=False).sum()
/var/folders/w4/fbz2zhr165g6wr5k09mtgl9r0000gn/T/ipykernel_66111/4184146311.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  n = geo_info[col].astype(str).str.contains(tok, case=False, regex=True, na=False).sum()


,gse_id,title,source_name_ch1,characteristics_ch1,cell_line
0,none,none,none,none,none
1,none,none,none,none,none
2,none,none,none,none,none
3,none,none,none,none,none
4,none,none,none,none,none
5,none,none,none,none,none
6,none,none,none,none,none
7,none,none,none,none,none
8,none,none,none,none,none
9,none,none,none,none,none


In [14]:
info = geo_info[geo_info['gse_id'] != 'none'].copy()
print("usable rows:", len(info), "/", len(geo_info),
      " lines:", info['cellosaurus_id'].nunique())

for gse, g in info.groupby('gse_id'):
    titles = g['title'].dropna().unique()[:2]
    srcs   = g['source_name_ch1'].dropna().unique()[:5]
    print(f"\n=== {gse}  n={len(g)}  lines={g['cellosaurus_id'].nunique()} ===")
    print("  title :", " | ".join(map(str, titles))[:160])
    print("  source:", " | ".join(map(str, srcs))[:220])

# what is the 478-sample line actually made of?
big = info['cellosaurus_id'].value_counts().index[0]
sub = info[info['cellosaurus_id'] == big]
print(f"\n{big}: {len(sub)} samples across series {sub['gse_id'].value_counts().to_dict()}")
display(sub[['gse_id','source_name_ch1','characteristics_ch1']].drop_duplicates().head(20))

usable rows: 2329 / 3267  lines: 738

=== gse10843  n=206  lines=134 ===
  title : replicate 1 for bt549 cell line | replicate 1 for cal851 cell line
  source: human cell line

=== gse10890  n=112  lines=47 ===
  title : replicate 1 for bt549 breast cell line | replicate 1 for cal851 breast cell line
  source: human cell line

=== gse12790  n=68  lines=52 ===
  title : mcf10a_null_vector_rep1 | mcf10a_null_vector_rep2
  source: mcf10a treated with null vector 24h | mcf10a treated with hras vector 24h | mcf10a treated with mek1 vector 24h | breast cancer

=== gse14315  n=58  lines=9 ===
  title : a549-aza-1 | a549-aza-2
  source: a549 cell line | h460 cell line | h125 cell line | zl25 cell line | calu3 cell line

=== gse15329  n=63  lines=39 ===
  title : human non-hodgkins lymphoma cell line, a3_kawakami_rep1 | human non-hodgkins lymphoma cell line, a3_kawakami_rep2
  source: human non-hodgkins lymphoma cell line

=== gse23806  n=80  lines=37 ===
  title : gsr neurospheres hamburg gs-1

,gse_id,source_name_ch1,characteristics_ch1
236,gse50811,zrt control,cell line: zrt
238,gse50811,zrt eribulin_24h,cell line: zrt
241,gse50811,zrt paclitaxel_24h,cell line: zrt
865,gse57083,az gene expression,cell line: c106
888,gse57083,az gene expression,cell line: cc20
1060,gse57083,az gene expression,cell line: l115clone1
1063,gse57083,az gene expression,cell line: lb61
1070,gse57083,az gene expression,cell line: lncapcasres
1071,gse57083,az gene expression,cell line: lncapcasresnewcastle
1092,gse57083,az gene expression,cell line: mcf7mdrpos


In [15]:
BASELINE = {'gse57083','gse34211','gse10843','gse7127','gse10890','gse15329',
            'gse23806','gse53798','gse41445','gse65216','gse84557','gse50451'}
CONTROL_ARM = {'gse50811','gse50830','gse50831'}   # keep control rows only
DROP = {'gse12790','gse14315','gse30240'}          # perturbation, covered elsewhere

info = geo_info[geo_info['gse_id'] != 'none'].copy()

baseline_rows = info[info['gse_id'].isin(BASELINE)]

ctrl_mask = info['source_name_ch1'].str.contains(r'control|ctrl|naive|untreated',
                                                 case=False, na=False)
control_rows = info[info['gse_id'].isin(CONTROL_ARM) & ctrl_mask]

kept = pd.concat([baseline_rows, control_rows])
kept = kept[kept['cellosaurus_id'] != 'none']      # must attach to a model_id

print("kept samples:", len(kept), " lines:", kept['cellosaurus_id'].nunique())

# sanity-check the control filter actually excluded the drug arms
dropped_drug = info[info['gse_id'].isin(CONTROL_ARM) & ~ctrl_mask]
print("\ndrug arms dropped (should be all eribulin/paclitaxel):")
print(dropped_drug['source_name_ch1'].str.extract(r'(control|eribulin|paclitaxel)',
      expand=False).value_counts(dropna=False))

kept samples: 1652  lines: 733

drug arms dropped (should be all eribulin/paclitaxel):
source_name_ch1
paclitaxel    201
eribulin      200
Name: count, dtype: int64


In [16]:
# 1. Per-line array counts after filtering — expect a sane distribution now,
#    no single line in the hundreds.
print(kept.groupby('cellosaurus_id').size().describe())
print("max arrays for one line:", kept.groupby('cellosaurus_id').size().max())

# 2. The gse23806 culture-format caveat: any model_id present as BOTH
#    neurosphere and adherent within that series?
g = kept[kept['gse_id'] == 'gse23806']
fmt = g['source_name_ch1'].str.contains('neurosphere', case=False, na=False)
dual = g.groupby('cellosaurus_id')['source_name_ch1'].apply(
    lambda s: s.str.contains('neurosphere', case=False, na=False).nunique() > 1)
print("gse23806 lines with mixed culture formats:", dual.sum())

count    733.000000
mean       2.253752
std        2.620016
min        1.000000
25%        1.000000
50%        1.000000
75%        3.000000
max       42.000000
dtype: float64
max arrays for one line: 42
gse23806 lines with mixed culture formats: 2


In [17]:
c = kept.groupby('cellosaurus_id').size()
top = c.idxmax()
sub = kept[kept['cellosaurus_id'] == top]
print(top, "→", len(sub), "arrays across series", sub['gse_id'].value_counts().to_dict())
display(sub[['source_name_ch1','cell_line','matching_type']].drop_duplicates().head(45))

cvcl_0031 → 42 arrays across series {'gse10890': 30, 'gse41445': 3, 'gse57083': 3, 'gse50811': 3, 'gse34211': 2, 'gse10843': 1}


,source_name_ch1,cell_line,matching_type
52,mcf7,mcf7,cello cell line synonym
1089,kudos 95 cell panel,mcf7,cello geo gsm
1090,az gene expression,mcf7,cello geo gsm
2143,human cell line,mcf7,cello geo gsm
2351,human cell line,mcf7,cello cell line synonym
3072,mcf7,mcf7,cello geo gsm
253,mcf-7 control,mcf-7,cello cell line name


In [18]:
g = kept[kept['gse_id'] == 'gse23806'].copy()
is_ns = g['source_name_ch1'].str.contains('neurosphere', case=False, na=False)
dual_ids = g.groupby('cellosaurus_id')['source_name_ch1'].apply(
    lambda s: s.str.contains('neurosphere', case=False, na=False).nunique() > 1)
dual_ids = dual_ids[dual_ids].index
# keep adherent, drop neurosphere, for those two models only
kept = kept[~((kept['gse_id']=='gse23806') & is_ns.reindex(kept.index, fill_value=False)
              & kept['cellosaurus_id'].isin(dual_ids))]

In [21]:
hpa = pd.read_parquet("/Users/musa.official/Documents/UoB-GeneTraceAI-25-26/data/parquet/data_clean/hpa_rna_clean.parquet")
hpa.head(3)

,gene,gene name,cell line,tpm,ptpm,ntpm
0,ensg00000000003,tspan6,143b,22.0,27.6,25.9
1,ensg00000000003,tspan6,22rv1,2.8,3.6,2.7
2,ensg00000000003,tspan6,23132/87,6.2,7.5,7.5


In [22]:
# 1. One row per (gene, cell line)? The three columns are normalisations,
#    NOT duplicate measurements — so this should be exactly 1.
dup = hpa.groupby(['gene','cell line']).size()
print("max rows per (gene, cell line):", dup.max(), " duplicated pairs:", (dup>1).sum())

# 2. Prove the columns are monotone renormalisations: within one line,
#    tpm vs ntpm rank correlation should be ~1.0 (identical ordering).
one = hpa[hpa['cell line'] == hpa['cell line'].iloc[0]]
print("within-line tpm vs ntpm Spearman:", one['tpm'].corr(one['ntpm'], method='spearman'))

# 3. Coverage: HPA lines -> model_id via cellosaurus id, against your 1,840 roster.
print("HPA lines:", hpa['cell line'].nunique(),
      " with cellosaurus id:", hpa_desc['cellosaurus id'].nunique())

max rows per (gene, cell line): 1  duplicated pairs: 0
within-line tpm vs ntpm Spearman: 0.9984911926787218
HPA lines: 1206  with cellosaurus id: 1199


In [23]:
missing = hpa_desc[hpa_desc['cellosaurus id'].isna() | (hpa_desc['cellosaurus id']=='')]
print(len(missing))
missing[['cell line','disease','cellosaurus id']]


0


,cell line,disease,cellosaurus id


In [24]:
dupe_ids = hpa_desc['cellosaurus id'].value_counts()
dupe_ids = dupe_ids[dupe_ids > 1]
print(dupe_ids)
hpa_desc[hpa_desc['cellosaurus id'].isin(dupe_ids.index)]\
    [['cell line','disease','disease subtype','cellosaurus id','primary/metastasis']]\
    .sort_values('cellosaurus id')

cellosaurus id
none    8
Name: count, dtype: int64


,cell line,disease,disease subtype,cellosaurus id,primary/metastasis
38,af22,uncategorized,none,none,none
45,asc2telo differentiated,uncategorized,none,none,none
68,bj htert+ sv40 large t+,uncategorized,none,none,none
69,bj htert+ sv40 large t+ rasg12v,uncategorized,none,none,none
331,hhstec,uncategorized,none,none,none
394,hskmc,uncategorized,none,none,none
892,podo/svtert152,non-cancerous,none,none,none
893,podo/tert256,non-cancerous,none,none,none
